# Export AIS Data to CSV

This notebook exports decoded AIS tracks from an AISdb database to CSV, using the built-in `aisdb.write_csv()` writer and, for cases where you only want a handful of columns, a custom writer built on the standard library `csv` module. It is the runnable companion of the [AIS Data to CSV](https://aisviz.gitbook.io/documentation/tutorials/ais-data-to-csv) GitBook page, which walks through the same steps in more narrative detail.

The bundled `example_data.db` SQLite database (Gulf of Maine, January 2022) stands in for a database you would build with the [Database Loading](1-database-loading.ipynb) tutorial, so every cell here runs unattended without any external service.

**What you will learn**

- How to query tracks from a database and hand them to `aisdb.write_csv()` for a fully column-ordered, sanitized CSV
- How to write a custom CSV with `csv.DictWriter` when you only want a subset of columns
- How the same query pattern applies to a PostgreSQL connection, guarded behind a flag so it does not run against placeholder credentials


In [ ]:
# %pip install aisdb
# Running on Google Colab? Uncomment the line above to install AISdb first.


In [ ]:
import csv
import os

import aisdb
from aisdb import SQLiteDBConn, DBQuery, DomainFromPoints
from datetime import datetime


## Exporting with the built-in writer

Connect to the database, run a query, and hand the resulting tracks to `aisdb.write_csv()`. The query and the write both need to happen inside the same `with` block, because `TrackGen` and `DBQuery.gen_qry()` are lazy generators that read from the database connection as they are consumed; once the block exits and the connection closes, iterating over `tracks` again fails.

`write_csv()` figures out the column order from the fields present on the first track, appends a `datetime` column derived from the epoch `time` values, and rounds numeric fields like `lon` and `lat` to a fixed number of decimals. Static fields such as `mmsi` and `maneuver` appear once per track, while dynamic fields such as `lon`, `lat`, `cog`, `sog`, `heading`, `rot`, and `utc_second` get one row per AIS message, tagged with a `Track_ID` column so rows belonging to the same track can be grouped back together after loading the file elsewhere.


In [ ]:
dbpath = "./example_data.db"
start_time = datetime.strptime("2022-01-01 00:00:00", "%Y-%m-%d %H:%M:%S")
end_time = datetime.strptime("2022-01-08 00:00:00", "%Y-%m-%d %H:%M:%S")

# Gulf of Maine bounding box covered by example_data.db
domain = DomainFromPoints(points=[(-68.5, 42.8)], radial_distances=[250000])

csv_path = "./output_sqlite.csv"

with SQLiteDBConn(dbpath=dbpath) as dbconn:
    qry = DBQuery(
        dbconn=dbconn,
        start=start_time,
        end=end_time,
        xmin=domain.boundary["xmin"], xmax=domain.boundary["xmax"],
        ymin=domain.boundary["ymin"], ymax=domain.boundary["ymax"],
        callback=aisdb.database.sqlfcn_callbacks.in_timerange_validmmsi,
    )
    tracks = aisdb.track_gen.TrackGen(qry.gen_qry(), decimate=False)
    aisdb.write_csv(tracks, csv_path)

print(f"All tracks have been written to {csv_path}")


Checking the first few lines confirms the writer produced a properly ordered, sanitized file.


In [ ]:
with open(csv_path) as f:
    for line in f.readlines()[:6]:
        print(line.rstrip())


## Writing a custom CSV

`write_csv()` writes every column it finds, which is usually what you want. When you would rather control the exact set of columns yourself, iterate the tracks manually and write rows with the standard library `csv` module.

Two things trip people up here, and both come from the static-versus-dynamic distinction in a track dictionary. `mmsi` and `maneuver` are static, one value per track, so they are not indexed. `lon`, `lat`, `cog`, `sog`, `heading`, `rot`, and `utc_second` are dynamic arrays with one entry per AIS message, so each needs to be indexed by position `i` inside the inner loop.


In [ ]:
headers = [
    "mmsi", "time", "lon", "lat", "cog", "sog",
    "utc_second", "heading", "rot", "maneuver",
]

custom_csv_path = "./custom_output.csv"

with SQLiteDBConn(dbpath=dbpath) as dbconn, open(custom_csv_path, mode="w", newline="") as file:
    qry = DBQuery(
        dbconn=dbconn,
        start=start_time,
        end=end_time,
        xmin=domain.boundary["xmin"], xmax=domain.boundary["xmax"],
        ymin=domain.boundary["ymin"], ymax=domain.boundary["ymax"],
        callback=aisdb.database.sqlfcn_callbacks.in_timerange_validmmsi,
    )
    tracks = aisdb.track_gen.TrackGen(qry.gen_qry(), decimate=False)

    writer = csv.DictWriter(file, fieldnames=headers)
    writer.writeheader()

    for track in tracks:
        # Fields listed in track['dynamic'] hold one value per position;
        # everything else is a per-vessel scalar and repeats on every row.
        for i in range(len(track["time"])):
            row = {}
            for key in headers:
                value = track[key]
                row[key] = value[i] if key in track["dynamic"] else value
            writer.writerow(row)

print(f"All tracks have been written to {custom_csv_path}")


In [ ]:
with open(custom_csv_path) as f:
    for line in f.readlines()[:6]:
        print(line.rstrip())


## The PostgreSQL variant

The pattern is identical for a PostgreSQL-backed database, only the connection object changes. `PostgresDBConn` accepts the same keyword arguments as `psycopg`, including `host`, `port`, `user`, `dbname`, and `password`, or a single `libpq_connstring`. Because this notebook runs unattended against the bundled SQLite file, the cell below is guarded behind `RUN_POSTGRES` and stays `False` by default; flip it to `True` and fill in real credentials once you have a PostgreSQL AISdb database to connect to.


In [ ]:
RUN_POSTGRES = False  # set to True and fill in real credentials to run this cell

if RUN_POSTGRES:
    from aisdb.database.dbconn import PostgresDBConn

    postgres_csv_path = "./output_postgresql.csv"

    with PostgresDBConn(
        host="localhost",             # PostgreSQL address
        port=5432,                    # PostgreSQL port
        user="your_username",         # PostgreSQL username
        password="your_password",     # PostgreSQL password
        dbname="database_name",       # database name
    ) as dbconn:
        qry = DBQuery(
            dbconn=dbconn,
            start=start_time,
            end=end_time,
            xmin=domain.boundary["xmin"], xmax=domain.boundary["xmax"],
            ymin=domain.boundary["ymin"], ymax=domain.boundary["ymax"],
            callback=aisdb.database.sqlfcn_callbacks.in_timerange_validmmsi,
        )
        tracks = aisdb.track_gen.TrackGen(qry.gen_qry(), decimate=False)
        aisdb.write_csv(tracks, postgres_csv_path)

    print(f"All tracks have been written to {postgres_csv_path}")
else:
    print("RUN_POSTGRES is False, skipping the PostgreSQL export.")


## Lower-level building blocks

If you are assembling rows yourself outside of `csv.DictWriter`, AISdb also exposes the two pieces `write_csv()` is built from, both importable from `aisdb.proc_util` rather than the top-level package. `tracks_csv(tracks, skipcols=['label', 'in_zone'])` is a generator that yields the header row followed by one already column-ordered, sanitized row per AIS message. `write_csv_rows(rows, pathname='output.csv', mode='a')` appends any iterable of row tuples to a file, which is useful when you are assembling rows from more than one source before writing them out.


## Cleaning up

The CSV files written in this notebook are scratch output for the demo, so remove them once you have finished inspecting the results.


In [ ]:
for path in (csv_path, custom_csv_path):
    if os.path.exists(path):
        os.remove(path)
        print(f"Removed {path}")


## Next steps

Continue with [9. Bathymetric Data](9-bathymetric-data.ipynb) to merge depth values onto tracks before exporting, or head back to [7. Using Your AIS Data](7-using-your-ais-data.ipynb) to build a database from your own AIS feed. See the [AIS Data to CSV](https://aisviz.gitbook.io/documentation/tutorials/ais-data-to-csv) GitBook page for the full narrative walkthrough.
